<a href="https://colab.research.google.com/github/rangaraju1/AI-agents/blob/main/financial_information_chat_agent_memory_management_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Objectives

- Implement Advanced Langgraph Memory: Explore LangMem for adding semantic, long-term memory capabilities to LangGraph agents. Understand how to configure LangMem stores (like InMemoryStore or persistent stores) and integrate its `search_memory` and `manage_memory` tools into the agent's workflow. Grasp the concept of using namespaces (like `thread_id`) to isolate memories between different conversation sessions.

- Build Context-Aware Conversational Agents: Grasp the principles of designing agent prompts (Task description) and workflows (Crew process) that instruct the agent to effectively use its memory and tools to answer user queries in a coherent, multi-turn conversation.

# Setup

In [ ]:
!pip install -q openai==1.66.3 \
                chromadb==0.6.3 \
                langchain==0.3.20 \
                langchain-openai==0.3.9 \
                langchain-chroma==0.2.2 \
                langgraph==0.4.1 \
                langmem==0.0.25

In [ ]:
import os
import uuid
import chromadb

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma

from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

from langmem import create_manage_memory_tool, create_search_memory_tool

from google.colab import userdata

In [ ]:
openai_api_key = userdata.get('openai_api_key')

llm = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=openai_api_key,
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    temperature=0
)

embedding_model = OpenAIEmbeddings(
    api_key=openai_api_key,
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    model="text-embedding-3-small"
)

# Business Scenario

A financial services firm needs to provide its analysts and portfolio managers with rapid, accurate, and context-aware insights derived from vast amounts of regulatory filings (like SEC 10-Ks) for various companies. Analysts often ask follow-up questions or refer back to previously discussed topics during their research sessions.

Problem: Manually searching through hundreds of pages of dense 10-K documents for specific figures, risk factors, or strategic statements is time-consuming and prone to missing context from previous queries within the same research session.

Solution: Implement an agent workflow that provides an AI assistant specialized in a specific company (e.g., Tesla).
- The analyst can ask direct questions ("What were Tesla's R&D expenses in 2022 according to the 10k?"), and the agent uses the retriever tool to fetch the relevant snippet.
- The analyst can ask follow-up questions ("What risks were mentioned alongside that?", "Remind me about the CEO you mentioned earlier?"). The agent uses short-term memory to remember the immediate conversational flow and Entity Memory to recall specific details about "Tesla" or "Elon Musk" mentioned previously in this session.
- Reduces time spent manually searching documents.
- Ensures answers are grounded in the official filings when required.
- Frees up the analyst's time for higher-level analysis rather than basic information gathering.
- (Potential) Over time, the long-term memory captures how well the agent performs on certain types of queries and might offer insights (via the stored suggestions/quality) into refining the agent's instructions or identifying common areas of inquiry.

# Implementation Roadmap

**Core Idea:**
For agents engaging in extended dialogues, like a research assistant answering queries over time, it's crucial to go beyond stateless responses. We need mechanisms to track the conversation's flow, remember key facts, figures, or entity details explicitly (handled by LangMem tools), and access external knowledge (like 10-K filings via a retriever). This allows the agent to provide more relevant, coherent answers and avoid repeatedly asking for or providing the same information, guided by explicit instructions in its system prompt.

**Implementation Logic:**
1.  *Foundation - External Knowledge:* Recognize the agent won't inherently know every detail from specific documents (Tesla 10k). Load these documents into a searchable vector database (ChromaDB) using text embeddings. This acts as the agent's external "filing cabinet".
2.  *Access Tool (Retriever):* Define a Python function (`retrieve_tesla_10k_data`) that uses a LangChain retriever to query the ChromaDB. Crucially, document this function with a clear docstring explaining its purpose and when the agent should use it (only for specific 10-K facts).
3.  *Memory Tools (LangMem):* Instantiate LangMem's `search_memory_tool` and `manage_memory_tool`, connecting them to a chosen store (InMemoryStore here). Configure them with a namespace template tied to LangGraph's `thread_id` to keep memories separate for each conversation.
4.  *The Agent's Brain (LLM & Prompt):* Initialize the LLM (ChatOpenAI). Craft a detailed System Prompt that defines the agent's persona (Expert Financial Analyst), its goal, lists the available tools (retriever, search memory, manage memory), and provides explicit step-by-step instructions on how to process user queries: *always* check memory first (`search_memory_tool`), then decide whether to use the retriever or general knowledge, synthesize the answer, and *finally* consider saving important information (`manage_memory_tool`).
5.  *Orchestration (LangGraph Agent):* Use `create_react_agent` from LangGraph to build the agent executor. Provide it with the LLM, the list of all tools (retriever + memory tools), the detailed System Prompt, and a `MemorySaver` instance for managing the basic chat history.
6.  *State Management (Conversation Thread):* Generate a unique `thread_id` for each distinct conversation session. Pass this ID in the `config` dictionary during every `agent.invoke` call. This ensures LangGraph's `MemorySaver` and LangMem's tools correctly associate messages and memories with the specific ongoing conversation.
7.  *Simulation & Observation:* Run a series of user questions through the `agent.invoke` method, passing the `thread_id` each time. Observe the agent's responses and (by examining the intermediate steps if verbose logging is enabled, or through the agent's ability to answer follow-up questions) verify that it uses the memory search and retriever tools as instructed by the system prompt.

# Vector Store Setup

This code block sets up access to the pre-existing Chroma vector database containing Tesla 10-K document embeddings.
- `! unzip tesla_db.zip`: A shell command (specific to environments like Colab) to unzip the ChromaDB data files into the './tesla_db' directory.
- `chromadb.PersistentClient`: Creates a client connection to the persistent ChromaDB stored on disk at the specified path.
- `chromadb_client.get_collection`: Attempts to access the specific collection within the database to ensure it exists.
- `Chroma(...)`: Creates a LangChain Chroma vector store object, linking it to the specific collection name, the initialized `embedding_model`, and the persistent `chromadb_client`. This object provides a LangChain-compatible interface to the vector store.
- `.as_retriever()`: Converts the LangChain vector store object into a Retriever object. The retriever is configured here to use similarity search and return the top 5 most relevant document chunks (`k=5`) for a given query. This retriever will be used by the `retrieve_tesla_10k_data` tool.

In [ ]:
! unzip tesla_db.zip

unzip:  cannot find or open tesla_db.zip, tesla_db.zip.zip or tesla_db.zip.ZIP.


In [ ]:
chromadb_client = chromadb.PersistentClient(
    path='./tesla_db'
)

tesla_10k_collection = 'tesla-10k-2019-to-2023'

try:
    chromadb_client.get_collection(tesla_10k_collection)
    print(f"Collection '{tesla_10k_collection}' found.")
except Exception as e:
    print(f"Warning: Collection '{tesla_10k_collection}' not found. Ensure './tesla_db' exists and is populated. Error: {e}")

vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    embedding_function=embedding_model,
    client=chromadb_client,
)

retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection 'tesla-10k-2019-to-2023' found.


# Retriever Tool Definition

This code block defines a standard Python function that acts as a tool for the LangGraph agent.
- The function `retrieve_tesla_10k_data` takes a string query as input.
- It uses the `retriever.invoke(query)` method (from the ChromaDB setup) to perform the vector search.
- It formats the retrieved document `page_content` into a single string, clearly indicating the source ("Excerpts from Tesla 10-K filings").

This function is later passed in as a tool to the`create_react_agent` function.

In [ ]:
def retrieve_tesla_10k_data(query: str) -> str:
    """
    Search and return relevant excerpts from Tesla 10K documents (2019-2023).
    Use this tool ONLY when the user asks for specific financial figures, operational details,
    reported risks, product specifics, or other factual data explicitly likely to be found
    in Tesla's official 10-K filings.
    Do NOT use this tool for general knowledge questions about Tesla (e.g., 'Who is the CEO?', 'What does Tesla do?').
    """
    print(f"--- Tool: retrieve_tesla_10k_data --- Input: {query=}")
    try:
        results = retriever.invoke(query)
        if not results:
            return "No specific information found in the 10-K excerpts for that query."
        final_output = "Excerpts from Tesla 10-K filings:\n"
        for doc in results:
            final_output += f"\n---\n{doc.page_content}\n---\n"
        return final_output
    except Exception as e:
        print(f"Error during retrieval: {e}")
        return "Error accessing or searching the 10-K document database."

# Memory Configuration

This section configures the memory components for the agent using LangMem and LangGraph's built-in capabilities.
- `InMemoryStore`: This initializes Langgraph's simplest storage backend. It holds semantic memories (embeddings and text) only for the duration of the script run. For persistence across runs, a file-based store like `LanceDBStore` or a database store like `RedisStore` would be used. The `index` configuration tells the store the dimensionality of the embeddings (1536 for text-embedding-3-small) and specifies the embedding model object to use for creating embeddings.
- `memory_namespace_template`: Defines a pattern for organizing memories. `("tesla_research_session", "{thread_id}")` means memories will be grouped under the "tesla_research_session" category, and further isolated by the unique `thread_id` of the conversation provided during agent invocation. This prevents memory leakage between different chat sessions.
- `create_manage_memory_tool`: Creates the LangMem tool responsible for *saving* information to the semantic memory store. The agent will be instructed (via the system prompt) to call this tool to record important facts. It's linked to the `store` and uses the `namespace` template.
- `create_search_memory_tool`: Creates the LangMem tool responsible for *retrieving* information from the semantic memory store based on semantic similarity to the query. The agent will be instructed to use this tool first to check for relevant context before using other tools. It's also linked to the `store` and `namespace` template.

Note: LangGraph's `MemorySaver` (initialized later) complements this by managing the chronological sequence of *all* messages (Human, AI, Tool) within a thread, providing basic conversational context, while LangMem tools handle the explicit saving and searching of specific, semantically-indexed facts.

In [ ]:
store = InMemoryStore(
    index={
        "dims": 1536, # Dimension for text-embedding-3-small
        "embed": embedding_model # Pass the initialized embedding model directly
    }
)

In [ ]:
# Define a namespace template. Use 'thread_id' which LangGraph uses for conversations.
# This keeps memories separate per conversation thread.
memory_namespace_template = ("tesla_research_session", "{thread_id}")

In [ ]:
# Create the memory management tools
manage_memory_tool = create_manage_memory_tool(
    store=store,
    namespace=memory_namespace_template
)

In [ ]:
search_memory_tool = create_search_memory_tool(
    store=store,
    namespace=memory_namespace_template
)

# Agent Initialization

This section assembles the agent using LangGraph's `create_react_agent`.
- `all_tools`: A list containing all the tools the agent can potentially use: the custom `retrieve_tesla_10k_data` function and the two LangMem tools (`search_memory_tool`, `manage_memory_tool`).
- `system_prompt`: A multi-line string containing detailed instructions for the agent. This is the core of the agent's configuration, defining its role, goal, available tools (and how/when to use them), and the step-by-step process for handling user queries, including the crucial memory interaction steps (search first, save later).
- `MemorySaver()`: Initializes LangGraph's built-in checkpointing mechanism. It automatically saves the state (sequence of messages) of the conversation thread after each step, allowing the conversation to be paused and resumed. It provides the basic turn-by-turn conversational context. It is distinct from the LangMem semantic memory but works alongside it.

In [ ]:
all_tools = [
    retrieve_tesla_10k_data,
    search_memory_tool,
    manage_memory_tool
]

In [ ]:
# --- System Prompt (Crucial for guiding the LangGraph agent) ---
# This combines the role, goal, backstory, task description, and memory instructions
system_prompt = """
You are an Expert Financial Analyst specializing in Tesla. Your goal is to provide insightful answers to questions about Tesla, leveraging company filings and your memory of this conversation.

You have access to several tools:
1.  `retrieve_tesla_10k_data`: Use this ONLY for specific facts, figures, risks, or operational details likely found in Tesla's 10-K filings (2019-2023). Do NOT use for general knowledge.
2.  `search_memory_tool`: Use this FIRST for EVERY user query to recall relevant context, facts, or entities mentioned earlier in THIS conversation session. Check memory before deciding to use other tools or your general knowledge.
3.  `manage_memory_tool`: Use this AFTER you have successfully answered a query, especially if you used the `retrieve_tesla_10k_data` tool or provided a key piece of information (like the CEO's name, a specific risk, a product detail). Save a concise summary of the question and answer, or the key fact retrieved, to remember it for later. Only save important, factual information.

Your process for answering each user query:
1.  **Check Memory:** ALWAYS start by using `search_memory_tool` with the user's query to see if relevant information or context already exists in our conversation history.
2.  **Analyze Query & Use Tools:**
    *   If memory provides the answer or sufficient context, use it.
    *   If the query asks for specific 10-K data AND memory wasn't enough, use `retrieve_tesla_10k_data`.
    *   If it's a general knowledge question (and not in memory), answer directly using your internal knowledge.
    *   DO NOT use `retrieve_tesla_10k_data` for general questions like 'Who is the CEO?'.
3.  **Synthesize Answer:** Combine information from your memory search results, any tool results (like 10-K excerpts), and your general knowledge to formulate a clear, concise, and accurate answer. Directly address the user's query. If you used the retriever, mention it (e.g., "According to the 10-K excerpts...").
4.  **Save to Memory:** After providing your answer, if it contained significant new information (especially from the retriever tool or key facts identified), use `manage_memory_tool` to save a concise summary (e.g., "User asked about 2022 R&D expenses. Found $X value in 10k." or "Confirmed Elon Musk is the CEO.").

Provide only the final answer to the user. Do not explicitly state you are searching or saving memory unless directly asked about your capabilities.
"""

In [ ]:
memory = MemorySaver()

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=system_prompt,
    checkpointer=memory, # Use the memory saver for checkpointing conversation history
    store=store
)

TypeError: StateGraph.add_node() got an unexpected keyword argument 'input_schema'

# Conversation Simulation

In [ ]:
conversation_thread_id = str(uuid.uuid4())

conversation_questions = [
    "What does Tesla primarily do?", # General knowledge, maybe save?
    "Who is the CEO of Tesla?", # General knowledge, should be saved to memory
    "Can you tell me about the main risks Tesla faces, according to their filings?", # Should trigger retriever & save
    "What did I just ask about?", # Should trigger memory search
    "What are the key vehicle products besides Model S and X?", # General knowledge / maybe retriever if specific years mentioned? Should save.
    "What are the development challenges mentioned for the Cybertruck in the 2023 10k?", # Should trigger retriever & save
    "Remind me, who leads the company we are discussing?", # Should use memory search
    "Summarize the main points we've discussed about Tesla's business and risks." # Should use memory search extensively
]

In [ ]:
results_log = []

In [ ]:
for i, question in enumerate(conversation_questions):
    print(f"\n--- Turn {i+1} ---")
    print(f"User Query: {question}")

    # Prepare input message
    messages_input = [HumanMessage(content=question)]

    # Define the configuration for this invocation, critical for memory
    config = {"configurable": {"thread_id": conversation_thread_id}}

    # Invoke the agent
    print("Agent Thinking...")
    response = agent.invoke({"messages": messages_input}, config=config)

    # Extract the final AI message
    final_ai_message = response['messages'][-1].content
    print(f"\nAgent Response:\n{final_ai_message}")

    results_log.append({
        "question": question,
        "answer": final_ai_message,
        "full_response": response # Store full response for potential debugging
    })
    print("--- Turn End ---")

In [ ]:
print("\n\n--- Conversation Summary ---")
for log in results_log:
    print(f"Q: {log['question']}")
    print(f"A: {log['answer']}")
    print("\n---\n")